# Inspection scenarios and sensitivity analysis 

Runs the three inspection scenarios (random, equal weight and adjusted weight) and the
sensitivity analysis of the deterioration step (0.05, 0.10, 0.15) and the improvement step (1, 2).
Each setting runs 10 times and reports the mean caught rate (± 95% CI),
saves the compliance distribution figures.


In [1]:
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
from math import sqrt
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector
from ml_inference import predict_inspection_result
from IPython.display import display, Image

## Settings

In [2]:
INPUT_FILE = "init_ABM_dataset.xlsx"

GRID_W, GRID_H = 90, 55
TICK_DAYS = 90
SIM_START_DATE = datetime.today().date()


N_FBO = 1000
N_TICKS = 20
N_REPS = 10
SCENARIOS = ("random", "equal", "adjusted")

# Inspector parameters
N_INSPECTORS = 8
MOVE_RANGE = 18
SIGHT = 20
INSPECTION_CAPACITY = 25

DELTA_GRID = (0.05, 0.10, 0.15)   # deterioration step
IMPROVE_GRID = (1, 2)             # improvement step

POP_SEED = 42   # fixed FBO agents sample

SCENARIO_CONFIG = {
    "random": {
        "weights": dict(size=0, former=0, type=0, product=0, interval=0),
        "random_flag": True,
    },
    "equal": {
        "weights": dict(size=1, former=1, type=1, product=1, interval=1),
        "random_flag": False,
    },
    "adjusted": {
        "weights": dict(size=0, former=1.5, type=-1, product=1, interval=2),
        "random_flag": False,
    },
}

# Outputs
OUTPUT_DIR = "outputs"
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figs")
RESULTS_FILE = os.path.join(OUTPUT_DIR, "scenario_and_sensitivity_results.xlsx")
os.makedirs(FIGURE_DIR, exist_ok=True)

## Load data

In [3]:
data = pd.read_excel(INPUT_FILE, parse_dates=["Last Inspection date"])


def load_rows_with_date(df, n, sim_start_date, sample_seed=POP_SEED):
    items = []
    for idx, r in df.sample(n, random_state=sample_seed).iterrows():
        last_date = r["Last Inspection date"].date()
        items.append({
            "agent_id": f"FBO_{idx}",
            "FBO type": r["FBO type"],
            "Product type": r["Product type"],
            "Operation type": r["Operation type"],
            "FBO Size": r["FBO Size"],
            "inspection_result": r["Inspection result"],
            "inspection_interval_days": max(0, (sim_start_date - last_date).days),
        })
    return items

## Agent-based Model

In [4]:
# FBO agents
class FBOAgent(Agent):
    def __init__(self, uid, model, info):
        super().__init__(uid, model)
        self.agent_id = info["agent_id"]
        self.FBO_type = info["FBO type"]
        self.Product_type = info["Product type"]
        self.Operation_type = info["Operation type"]
        self.FBO_size = info["FBO Size"]
        self.inspection_result = info["inspection_result"]

        self.compliance_level = self.inspection_result
        self.actual_compliance_level = self.compliance_level   # latent continuous score
        self.former_inspection_result = self.inspection_result
        self.inspection_interval_days = info["inspection_interval_days"]

    def features(self):
        return {
            "FBO type": self.FBO_type,
            "Product type": self.Product_type,
            "Operation type": self.Operation_type,
            "Former Inspection result": self.former_inspection_result,
            "FBO Size": self.FBO_size,
            "Inspection interval (days)": self.inspection_interval_days,
        }

    def pre_ml_tick_update(self):
        # possible deterioration, for FBOs not inspected in the most recent tick
        ch = predict_inspection_result(self.features())
        if ch == 1 and self.inspection_interval_days != TICK_DAYS:
            self.actual_compliance_level = min(4, self.actual_compliance_level + self.model.deterioration_step)
            self.compliance_level = round(self.actual_compliance_level)

    def inspect_and_maybe_improve(self):
        self.inspection_result = self.compliance_level
        if self.compliance_level > 1:
            self.model.at_risk_inspected_this_tick += 1

        ch = predict_inspection_result(self.features())
        if ch == -1:
            self.compliance_level = max(1, self.compliance_level - self.model.improvement_step)

        self.actual_compliance_level = self.compliance_level
        self.inspection_interval_days = TICK_DAYS
        self.former_inspection_result = self.inspection_result
       
# Inspector agents
class InspectorAgent(Agent):
    def __init__(self, uid, model, move_range, sight, inspection_capacity):
        super().__init__(uid, model)
        self.move_range = move_range
        self.sight = sight
        self.inspection_capacity = inspection_capacity

    def step(self):
        # random move, try to land on an empty cell
        for _ in range(12):
            dx = random.randint(-self.move_range, self.move_range)
            dy = random.randint(-self.move_range, self.move_range)
            nx = (self.pos[0] + dx) % self.model.grid.width
            ny = (self.pos[1] + dy) % self.model.grid.height
            if self.model.grid.is_cell_empty((nx, ny)):
                break
        self.model.grid.move_agent(self, (nx, ny))

        nb = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False, radius=self.sight)
        fbos = [a for a in nb if isinstance(a, FBOAgent)]
        if not fbos:
            return

        # rescale value of each factor to the same range 
        def scale(arr):
            a = np.array(arr, dtype=float)
            mn, mx = a.min(), a.max()
            return np.full_like(a, 2.5) if mx == mn else 1 + 3 * (a - mn) / (mx - mn)

        sizes = scale([f.FBO_size for f in fbos])
        formers = scale([f.former_inspection_result for f in fbos])
        types = scale([f.FBO_type for f in fbos])
        products = scale([f.Product_type for f in fbos])
        intervals = scale([f.inspection_interval_days for f in fbos])

        w = self.model.weights
        score = (
            w["size"] * sizes
            + w["former"] * formers
            + w["type"] * types
            + w["product"] * products
            + w["interval"] * intervals
        )

        # all weights 0 is treated as random inspection
        random_condition = self.model.random_inspection or all(v == 0 for v in w.values())
        order = np.random.permutation(len(fbos)) if random_condition else np.argsort(-score)

        # Select FBOs to be inspected
        selected = []
        for idx in order:
            f = fbos[idx]
            if f not in self.model.assigned_this_tick:
                selected.append(f)
                self.model.assigned_this_tick.add(f)
            if len(selected) >= self.inspection_capacity:
                break

        for f in selected:
            f.inspect_and_maybe_improve()

# Model
class InspectionModel(Model):
    def __init__(
        self, items, num_inspectors=8, width=GRID_W, height=GRID_H,
        w_size=1.0, w_former=1.0, w_type=1.0, w_product=1.0, w_interval=1.0,
        move_range=18, sight=20, inspection_capacity=25, random_inspection=False,
        deterioration_step=0.10, improvement_step=1
    ):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=True)
        self.schedule = RandomActivation(self)

        self.weights = {
            "size": w_size, "former": w_former, "type": w_type,
            "product": w_product, "interval": w_interval
        }
        self.random_inspection = random_inspection
        # Sensitivity parameters
        self.deterioration_step = deterioration_step
        self.improvement_step = improvement_step

        self.assigned_this_tick = set()
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

         # FBOs 
        for info in items:
            f = FBOAgent(info["agent_id"], self, info)
            x, y = self.random.randrange(width), self.random.randrange(height)
            self.grid.place_agent(f, (x, y))
            self.schedule.add(f)

        # inspectors
        for i in range(num_inspectors):
            ins = InspectorAgent(f"INS_{i}", self, move_range, sight, inspection_capacity)
            while True:
                x, y = self.random.randrange(width), self.random.randrange(height)
                if self.grid.is_cell_empty((x, y)):
                    break
            self.grid.place_agent(ins, (x, y))
            self.schedule.add(ins)

        # DataCollector
        self.datacollector = DataCollector(model_reporters={
            "compliance_level_1": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 1),
            "compliance_level_2": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 2),
            "compliance_level_3": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 3),
            "compliance_level_4": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 4),

            "at_risk_start": lambda m: m.at_risk_this_tick,

            "caught_rate": lambda m: (
                0.0 if m.at_risk_this_tick == 0
                else 100.0 * m.at_risk_inspected_this_tick / m.at_risk_this_tick
            ),
        })

    def step(self):
        # Reset tick trackers
        self.assigned_this_tick.clear()
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

        # Pre-inspection update 
        for a in self.schedule.agents:
            if isinstance(a, FBOAgent):
                a.pre_ml_tick_update()

        self.at_risk_this_tick = sum(
            1 for a in self.schedule.agents
            if isinstance(a, FBOAgent) and a.compliance_level > 1
        )

        # Inspectors
        for a in self.schedule.agents:
            if isinstance(a, InspectorAgent):
                a.step()

        # interval update for non-inspected
  
        for a in self.schedule.agents:
            if isinstance(a, FBOAgent) and a not in self.assigned_this_tick:
                a.inspection_interval_days += TICK_DAYS

        self.datacollector.collect(self)

## Run ABM

In [5]:
# one simulation run (20 ticks) for one scenario and one parameter setting;
# returns the mean caught rate and the number of FBOs per compliance level per tick
def run_single(run_seed, scenario, n_fbo=N_FBO, deterioration_step=0.10, improvement_step=1, sample_seed=POP_SEED):
    random.seed(run_seed)      # inspector movement
    np.random.seed(run_seed)   # order in random inspection

    items = load_rows_with_date(data, n_fbo, SIM_START_DATE, sample_seed)
    w = SCENARIO_CONFIG[scenario]["weights"]
    random_flag = SCENARIO_CONFIG[scenario]["random_flag"]

    m = InspectionModel(
        items,
        num_inspectors=N_INSPECTORS,
        w_size=w["size"], w_former=w["former"], w_type=w["type"],
        w_product=w["product"], w_interval=w["interval"],
        move_range=MOVE_RANGE, sight=SIGHT, inspection_capacity=INSPECTION_CAPACITY,
        random_inspection=random_flag,
        deterioration_step=deterioration_step,
        improvement_step=improvement_step,
    )
    for _ in range(N_TICKS):
        m.step()

    df = (
        m.datacollector
         .get_model_vars_dataframe()
         .reset_index(drop=False)
         .rename(columns={"index": "tick"})
    )

    # mean caught rate, excluding ticks without non-compliant FBOs
    caught_rate_mean = df.loc[df["at_risk_start"] > 0, "caught_rate"].mean()

    ts = df[["tick", "compliance_level_1", "compliance_level_2",
             "compliance_level_3", "compliance_level_4"]]
    return caught_rate_mean, ts

## Calculate CI across 10 runs

In [6]:
def mean_ci(vals):
    a = np.asarray(vals, float)
    half = 1.96 * a.std(ddof=1) / sqrt(len(a)) # 95% CI
    return a.mean(), half

## SA

In [7]:
def run_sensitivity_delta(delta_grid=DELTA_GRID, scenarios=SCENARIOS, n_reps=N_REPS, n_fbo=N_FBO):
    # change deterioration step 
    rows = []
    ts_store = {(d, sc): [] for d in delta_grid for sc in scenarios}

    for d in delta_grid:
        for sc in scenarios:
            caught_rates = []
            for r in range(n_reps):
                run_seed = 1000 + 37*r + int(d*100)
                caught_rate_mean, ts = run_single(run_seed, sc, n_fbo=n_fbo,
                                                  deterioration_step=d, improvement_step=1)
                caught_rates.append(caught_rate_mean)
                ts_store[(d, sc)].append(ts)

            m, ci = mean_ci(caught_rates)
            rows.append({
                "deterioration_step": d,
                "scenario": sc,
                "caught_rate_mean_%": round(m, 1),
                "ci_caught_rate_%": round(ci, 1),
            })

    return pd.DataFrame(rows), ts_store


def run_sensitivity_improve(improve_grid=IMPROVE_GRID, scenarios=SCENARIOS, n_reps=N_REPS, n_fbo=N_FBO):
    # change improvement step 
    rows = []
    ts_store = {(step, sc): [] for step in improve_grid for sc in scenarios}

    for step in improve_grid:
        for sc in scenarios:
            caught_rates = []
            for r in range(n_reps):
                run_seed = 2000 + 41*r + step
                caught_rate_mean, ts = run_single(run_seed, sc, n_fbo=n_fbo,
                                                  deterioration_step=0.10, improvement_step=step)
                caught_rates.append(caught_rate_mean)
                ts_store[(step, sc)].append(ts)

            m, ci = mean_ci(caught_rates)
            rows.append({
                "improvement_step": step,
                "scenario": sc,
                "caught_rate_mean_%": round(m, 1),
                "ci_caught_rate_%": round(ci, 1),
            })

    return pd.DataFrame(rows), ts_store

## Plot and tables

In [8]:
def mean_ci_table(df, index, columns, mean_col, ci_col):
    tmp = df.copy()
    tmp["value"] = tmp[mean_col].astype(str) + " ± " + tmp[ci_col].astype(str)
    return tmp.pivot_table(index=index, columns=columns, values="value", aggfunc="first")


def plot_three_scenarios_compliance(reps_by_scenario, param_label, param_value):
    levels = ["compliance_level_1", "compliance_level_2", "compliance_level_3", "compliance_level_4"]
    colors = ["green", "yellow", "orange", "red"]
    labels = ["Level 1", "Level 2", "Level 3", "Level 4"]
    letters = ["A", "B", "C"]
    alphas = np.linspace(0.25, 1.0, N_REPS)   

    fig, axes = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(6, 10))

    for ax, sc, letter in zip(axes, SCENARIOS, letters):
        for level, color in zip(levels, colors):
            for run, ts in enumerate(reps_by_scenario[sc]):
                ax.plot(ts["tick"], ts[level], color=color, alpha=alphas[run], linewidth=1.1)

        ax.set_ylabel("Number of FBOs")
        ax.set_ylim(0, N_FBO * 1.05)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title(f"{sc} scenario", loc="left", fontsize=10)
        ax.text(0.01, 0.95, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")

    axes[-1].set_xlabel("Tick")
    fig.suptitle(f"Compliance distributions — {param_label} = {param_value}", fontsize=12)

    legend_lines = [Line2D([0], [0], color=c, lw=2) for c in colors]
    fig.legend(legend_lines, labels, title="Compliance level",
               loc="center left", bbox_to_anchor=(1.03, 0.5), borderaxespad=0.0)
    fig.tight_layout(rect=(0, 0, 0.80, 0.95))

    safe_val = str(param_value).replace(".", "p")
    filename = os.path.join(FIGURE_DIR, f"compliance_{param_label}_{safe_val}.png")
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return filename

## Run model

In [ ]:
df_delta, ts_delta = run_sensitivity_delta()
df_improve, ts_improve = run_sensitivity_improve()

delta_table = mean_ci_table(df_delta, index="scenario", columns="deterioration_step",
                            mean_col="caught_rate_mean_%", ci_col="ci_caught_rate_%")
improve_table = mean_ci_table(df_improve, index="scenario", columns="improvement_step",
                              mean_col="caught_rate_mean_%", ci_col="ci_caught_rate_%")

## Output

In [ ]:
# caught-rate tables
print("Caught rate, deterioration step (mean ± 95% CI)")
display(delta_table)
print("Caught rate, improvement step (mean ± 95% CI)")
display(improve_table)

# compliance level figures 
for d in DELTA_GRID:
    reps_by_scenario = {sc: ts_delta[(d, sc)] for sc in SCENARIOS}
    filename = plot_three_scenarios_compliance(reps_by_scenario, "deterioration_step", d)
    display(Image(filename=filename))

for step in IMPROVE_GRID:
    reps_by_scenario = {sc: ts_improve[(step, sc)] for sc in SCENARIOS}
    filename = plot_three_scenarios_compliance(reps_by_scenario, "improvement_step", step)
    display(Image(filename=filename))


with pd.ExcelWriter(RESULTS_FILE) as writer:
    df_delta.to_excel(writer, sheet_name="delta_summary", index=False)
    df_improve.to_excel(writer, sheet_name="improve_summary", index=False)

    for (d, sc), reps in ts_delta.items():
        for i, df_ts in enumerate(reps, start=1):
            df_ts.to_excel(writer, sheet_name=f"d_{str(d).replace('.', 'p')}_{sc}_{i}", index=False)

    for (step, sc), reps in ts_improve.items():
        for i, df_ts in enumerate(reps, start=1):
            df_ts.to_excel(writer, sheet_name=f"im_{step}_{sc}_{i}", index=False)

print(f"Excel file written: {RESULTS_FILE}")